# LayoutVLM 完整复现 - CVPR 2025

**论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)

**GitHub**: https://github.com/sunfanyunn/LayoutVLM

---

## 🎯 功能说明

此notebook整合了完整的工作流程：
1. ✅ 在Colab中安装Blender 4.2.1
2. ✅ 配置GPU渲染
3. ✅ 运行LayoutVLM生成3D布局
4. ✅ 自动渲染可视化结果

---

## 📋 使用步骤

1. **启用GPU**: 运行时 → 更改运行时类型 → T4 GPU
2. **高RAM（推荐）**: 运行时 → 更改运行时类型 → 高RAM
3. **按顺序执行**所有单元格
4. **配置API**: 在步骤8中填入你的API密钥

⏱️ **预计时间**: 首次运行约15-20分钟（后续约5-10分钟）

---

## 步骤 1️⃣: 检查GPU环境

In [13]:
# 检查GPU
print('='*60)
print('🔍 检查GPU环境')
print('='*60)

!nvidia-smi

print('\n' + '='*60)
print('✅ GPU检查完成')
print('='*60)
print('\n⚠️  如果看不到GPU信息，请检查运行时设置')

🔍 检查GPU环境
Mon Nov  3 06:43:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-------------------------------------

## 步骤 2️⃣: 挂载Google Drive

💾 用于永久保存数据集和结果

In [14]:
from google.colab import drive
import os

# 挂载Drive
drive.mount('/content/drive')

# 创建项目目录结构
PROJECT_DIR = '/content/drive/MyDrive/LayoutVLM_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/blender_scripts', exist_ok=True)

print('\n' + '='*60)
print('✅ Google Drive已挂载')
print('='*60)
print(f'📁 项目目录: {PROJECT_DIR}')
print(f'📊 结果目录: {PROJECT_DIR}/results')
print(f'💾 数据集目录: {PROJECT_DIR}/datasets')
print('='*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive已挂载
📁 项目目录: /content/drive/MyDrive/LayoutVLM_Project
📊 结果目录: /content/drive/MyDrive/LayoutVLM_Project/results
💾 数据集目录: /content/drive/MyDrive/LayoutVLM_Project/datasets


## 步骤 3️⃣: 安装Blender 4.2.1

🔧 使用官方二进制包，支持完整的Python API

In [15]:
import os

print('='*60)
print('🔧 安装Blender 4.2.1')
print('='*60)

# 1. 安装系统依赖
print('\n📦 步骤1/4: 安装系统依赖...')
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install -y -qq xvfb libegl1-mesa libxrandr2 libxinerama1 libxxf86vm1 libxi6 wget > /dev/null 2>&1
print('   ✅ 系统依赖安装完成')

# 2. 下载Blender
print('\n📥 步骤2/4: 下载Blender 4.2.1...')
BLENDER_URL = "https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz"
BLENDER_FILE = "blender-4.2.1-linux-x64.tar.xz"

if not os.path.exists(f'/content/{BLENDER_FILE}'):
    !wget -q --show-progress {BLENDER_URL}
    print('   ✅ 下载完成')
else:
    print('   ✅ Blender压缩包已存在')

# 3. 解压Blender
print('\n📂 步骤3/4: 解压Blender...')
if not os.path.exists('/content/blender-4.2.1-linux-x64'):
    !tar -xJf {BLENDER_FILE}
    print('   ✅ 解压完成')
else:
    print('   ✅ Blender已解压')

# 4. 设置环境变量
BLENDER = "/content/blender-4.2.1-linux-x64/blender"
os.environ['BLENDER_PATH'] = BLENDER

# 5. 验证安装
print('\n🔍 步骤4/4: 验证Blender安装...')
!$BLENDER --version

print('\n' + '='*60)
print('✅ Blender 4.2.1 安装完成')
print(f'📍 位置: {BLENDER}')
print('='*60)

🔧 安装Blender 4.2.1

📦 步骤1/4: 安装系统依赖...
   ✅ 系统依赖安装完成

📥 步骤2/4: 下载Blender 4.2.1...
   ✅ Blender压缩包已存在

📂 步骤3/4: 解压Blender...
   ✅ Blender已解压

🔍 步骤4/4: 验证Blender安装...
Blender 4.2.1 LTS
	build date: 2024-08-19
	build time: 23:32:23
	build commit date: 2024-08-19
	build commit time: 11:21
	build hash: 396f546c9d82
	build branch: blender-v4.2-release
	build platform: Linux
	build type: Release
	build c flags:  -Wall -Werror=implicit-function-declaration -Wstrict-prototypes -Werror=return-type -Werror=vla -Wmissing-prototypes -Wno-char-subscripts -Wno-unknown-pragmas -Wpointer-arith -Wunused-parameter -Wwrite-strings -Wlogical-op -Wundef -Winit-self -Wmissing-include-dirs -Wno-div-by-zero -Wtype-limits -Wformat-signedness -Wrestrict -Wno-stringop-overread -Wno-stringop-overflow -Wnonnull -Wabsolute-value -Wuninitialized -Wredundant-decls -Wshadow -Wimplicit-fallthrough=5 -Wno-error=unused-but-set-variable  -march=x86-64-v2 -std=gnu11 -pipe -fPIC -funsigned-char -fno-strict-aliasing -ffp-contr

## 步骤 4️⃣: 创建GPU配置脚本

⚡ 启用GPU加速渲染

In [16]:
# 创建GPU配置脚本（基于你的set_cycles_gpu.py）
gpu_script = '''import bpy

# 配置Cycles渲染引擎使用GPU
prefs = bpy.context.preferences.addons["cycles"].preferences

# 尝试OPTIX，如果不支持则使用CUDA
try:
    prefs.compute_device_type = "OPTIX"
    print("✅ 使用OPTIX")
except Exception:
    prefs.compute_device_type = "CUDA"
    print("✅ 使用CUDA")

# 获取所有可用设备
prefs.get_devices()

# 启用所有GPU
gpu_count = 0
for dev in prefs.devices:
    try:
        dev.use = True
        if dev.type in ["CUDA", "OPTIX"]:
            gpu_count += 1
            print(f"   GPU {gpu_count}: {dev.name}")
    except:
        pass

# 设置场景使用GPU
bpy.context.scene.cycles.device = "GPU"

print(f"\\n[Cycles] 计算设备类型: {prefs.compute_device_type}")
print(f"[Cycles] 启用GPU数量: {gpu_count}")
print("✅ GPU渲染配置完成")
'''

# 保存到本地和Drive
with open('/content/set_cycles_gpu.py', 'w') as f:
    f.write(gpu_script)

import shutil
shutil.copy('/content/set_cycles_gpu.py', f'{PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')

print('='*60)
print('✅ GPU配置脚本已创建')
print('='*60)
print('📄 本地: /content/set_cycles_gpu.py')
print(f'💾 备份: {PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')
print('='*60)

✅ GPU配置脚本已创建
📄 本地: /content/set_cycles_gpu.py
💾 备份: /content/drive/MyDrive/LayoutVLM_Project/blender_scripts/set_cycles_gpu.py


## 步骤 5️⃣: 克隆LayoutVLM

📥 从GitHub获取最新代码

In [17]:
import os

os.chdir('/content')

print('='*60)
print('📥 克隆LayoutVLM仓库')
print('='*60)

# 清理旧版本
if os.path.exists('LayoutVLM'):
    print('\n🗑️  清理旧版本...')
    !rm -rf LayoutVLM

# 克隆仓库
print('\n📦 正在克隆...')
!git clone -b colab-compatibility https://github.com/HUMBLEDDDD/LayoutVLM.git

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ LayoutVLM已克隆')
print('='*60)
print(f'📍 位置: {os.getcwd()}')
print('\n📂 项目结构:')
!ls -1

📥 克隆LayoutVLM仓库

📦 正在克隆...
Cloning into 'LayoutVLM'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 132 (delta 27), reused 117 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 295.16 KiB | 8.68 MiB/s, done.
Resolving deltas: 100% (27/27), done.

✅ LayoutVLM已克隆
📍 位置: /content/LayoutVLM

📂 项目结构:
benchmark_tasks
main.py
prompts
README.md
requirements.txt
src
third_party
utils


In [18]:
with open('/content/LayoutVLM/src/layoutvlm/sandbox.py', 'r') as f:
    lines = f.readlines()
    for i in range(95, 110):  # 查看第96-110行
        print(f"{i:3d} | {repr(lines[i])}")

 95 | '        try:\n'
 96 | '            exec(self.all_code, _local_vars)\n'
 97 | '            exec(entire_program, _local_vars)\n'
 98 | '        except Exception as e:\n'
 99 | '            assert False, f"Error in the sandbox code: {e}"\n'
100 | '\n'
101 | '        for var_name, asset in _local_vars.items():\n'
102 | '            if type(asset).__name__ == "Assets":\n'
103 | '                for instance in asset.placements:\n'
104 | '                    assert instance.instance_id is not None\n'
105 | '                    assert instance.position is not None\n'
106 | '                    assert instance.rotation is not None\n'
107 | '            if type(asset).__name__ == "Walls":\n'
108 | '                for wall in asset.walls:\n'
109 | '                    assert wall.instance_id is not None\n'


## 步骤 6️⃣: 安装Python依赖到Blender

📦 **关键步骤**: 将依赖包安装到Blender的Python环境

In [19]:
import os

BLENDER = os.environ['BLENDER_PATH']

print('='*60)
print('📦 安装Python依赖到Blender')
print('='*60)
print('\n⏱️  这可能需要3-5分钟，请耐心等待...\n')

# 创建依赖安装脚本
install_script = '''import sys
import subprocess

# 需要安装的包
packages = [
# AI/LLM
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-core",
    "langchain-community",
    "tiktoken",
# 数值计算
    "numpy",
    "scipy",
# 深度学习
    "torch",
    "torchvision",
    "transformers",
    "accelerate",
# 3D/几何处理
    "trimesh",
    "shapely",
# 图像处理
    "Pillow",
    "imageio",
    "opencv-python",
# 工具
    "pyyaml",
    "tqdm",
    "requests",
# 可视化
    "matplotlib",
# 数据验证
    "pydantic",

]

print(f"Python路径: {sys.executable}")
print(f"Python版本: {sys.version.split()[0]}")
print(f"\\n开始安装 {len(packages)} 个依赖包...\\n")

success = 0
failed = []

for i, package in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] 安装 {package}...", end=" ")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("✅")
        success += 1
    except Exception as e:
        print(f"❌ ({str(e)[:30]}...)")
        failed.append(package)

print("\\n" + "="*60)
print(f"✅ 成功: {success}/{len(packages)}")
if failed:
    print(f"❌ 失败: {len(failed)} - {', '.join(failed)}")
print("="*60)
'''

# 保存安装脚本
with open('/tmp/install_deps_blender.py', 'w') as f:
    f.write(install_script)

# 使用Blender的Python执行安装
!$BLENDER --background --python /tmp/install_deps_blender.py

print('\n' + '='*60)
print('✅ 依赖安装完成')
print('='*60)

📦 安装Python依赖到Blender

⏱️  这可能需要3-5分钟，请耐心等待...

Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)
Python路径: /content/blender-4.2.1-linux-x64/4.2/python/bin/python3.11
Python版本: 3.11.7

开始安装 22 个依赖包...

[1/22] 安装 openai... ✅
[2/22] 安装 langchain... ✅
[3/22] 安装 langchain-openai... ✅
[4/22] 安装 langchain-core... ✅
[5/22] 安装 langchain-community... ✅
[6/22] 安装 tiktoken... ✅
[7/22] 安装 numpy... ✅
[8/22] 安装 scipy... ✅
[9/22] 安装 torch... ✅
[10/22] 安装 torchvision... ✅
[11/22] 安装 transformers... ✅
[12/22] 安装 accelerate... ✅
[13/22] 安装 trimesh... ✅
[14/22] 安装 shapely... ✅
[15/22] 安装 Pillow... ✅
[16/22] 安装 imageio... ✅
[17/22] 安装 opencv-python... ✅
[18/22] 安装 pyyaml... ✅
[19/22] 安装 tqdm... ✅
[20/22] 安装 requests... ✅
[21/22] 安装 matplotlib... ✅
[22/22] 安装 pydantic... ✅

✅ 成功: 22/22

Blender quit

✅ 依赖安装完成


## 步骤 7️⃣: 编译CUDA扩展

⚙️ 编译Rotated IOU Loss（用于边界框优化）

In [20]:
import os

print('='*60)
print('⚙️  编译CUDA扩展')
print('='*60)

os.chdir('/content/LayoutVLM/third_party/Rotated_IoU/cuda_op')

print('\n正在编译（可能需要1-2分钟）...\n')
!python setup.py install -q 2>&1 | grep -E "(Building|Installing|Finished|error|Error)"

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ CUDA扩展编译完成')
print('='*60)

⚙️  编译CUDA扩展

正在编译（可能需要1-2分钟）...

x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.12/dist-packages/torch/include -I/usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.12 -c sort_vert.cpp -o build/temp.linux-x86_64-cpython-312/sort_vert.o -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1018\" -DTORCH_EXTENSION_NAME=sort_vertices -std=c++17
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/sort_vert.o build/temp.linux-x86_64-cpython-312/sort_vert_kernel.o -L/usr/local/lib/python3.12

## 步骤 8️⃣: 准备数据集

💾 下载Objaverse资产数据集（约2.4GB）

In [21]:
import os

DATASET_PATH = f'{PROJECT_DIR}/datasets/dataset.zip'

print('='*60)
print('💾 准备数据集')
print('='*60)

# 检查Drive中是否已有数据集
if os.path.exists(DATASET_PATH):
    print('\n✅ 数据集已存在于Drive，直接复制...')
    !cp {DATASET_PATH} /content/LayoutVLM/dataset.zip
    print('   ✅ 复制完成')
else:
    print('\n📥 首次运行，正在下载数据集（约2.4GB）...')
    print('   这可能需要3-5分钟...\n')

    !pip install -q gdown
    !gdown 1WGbj8gWn-f-BRwqPKfoY06budBzgM0pu -O /content/LayoutVLM/dataset.zip

    print('\n💾 备份数据集到Drive（下次运行更快）...')
    !cp /content/LayoutVLM/dataset.zip {DATASET_PATH}
    print('   ✅ 备份完成')

# 解压数据集
print('\n📂 解压数据集...')
!mkdir -p /content/LayoutVLM/data
!unzip -q /content/LayoutVLM/dataset.zip -d /content/LayoutVLM/data/

# 检查解压结果
print('\n📊 数据集内容:')
!ls -lh /content/LayoutVLM/data/ | head -10

print('\n' + '='*60)
print('✅ 数据集准备完成')
print('='*60)

💾 准备数据集

✅ 数据集已存在于Drive，直接复制...
   ✅ 复制完成

📂 解压数据集...

📊 数据集内容:
total 40K
drwxr-xr-x 677 root root 36K Jun 18 17:46 test_asset_dir

✅ 数据集准备完成


## 步骤 9️⃣: 配置转接API

🔑 **重要**: 填入你的API密钥和配置

In [22]:
import os

print('='*60)
print('🔑 配置转接API')
print('='*60)

# ⚠️⚠️⚠️ 在这里填入你的配置 ⚠️⚠️⚠️
API_KEY = "sk-VdMqCoTjVMbI10EHlpb0DYWL8kBWCUbs7mUri5gT2obD6MgG"  # 替换成你的API密钥！
BASE_URL = "https://chat.cloudapi.vip/v1/"
MODEL_NAME = "gpt-4o"  # 或其他支持vision的模型

# 设置环境变量
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ['OPENAI_BASE_URL'] = BASE_URL

print('\n📋 API配置信息:')
print(f'   🔑 API Key: {API_KEY[:15]}...')
print(f'   🌐 Base URL: {BASE_URL}')
print(f'   🤖 Model: {MODEL_NAME}')

# 测试API连接
print('\n🧪 测试API连接...')
try:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "测试"}],
        max_tokens=5
    )
    print('   ✅ API连接成功')
    print(f'   📨 响应: {response.choices[0].message.content}')
except Exception as e:
    print(f'   ❌ API测试失败: {e}')
    print('   ⚠️  请检查API Key和Base URL是否正确')

print('\n' + '='*60)
print('✅ API配置完成')
print('='*60)

🔑 配置转接API

📋 API配置信息:
   🔑 API Key: sk-VdMqCoTjVMbI...
   🌐 Base URL: https://chat.cloudapi.vip/v1/
   🤖 Model: gpt-4o

🧪 测试API连接...
   ✅ API连接成功
   📨 响应: 你好！有什么我可以帮你的吗？

✅ API配置完成


## 步骤 🔟: 创建场景配置

📝 定义要生成的3D场景

In [23]:
import json
import os

os.chdir('/content/LayoutVLM')

print('='*60)
print('📝 创建场景配置')
print('='*60)

# 创建示例场景
scene_config = {
    "task_description": "a beach-inspired bedroom with light paint, a rattan chair and a queen bed, 4m x 5m",
    "layout_criteria": "airy, open, and serene with a focus on natural light and maximizing the feeling of space.\n",
    "boundary": {
        "floor_vertices": [
            [0, 0, 0],
            [4, 0, 0],
            [4, 5, 0],
            [0, 5, 0]
        ],
        "wall_height": 1.8687612077960416
    },
    "assets": {
        "ed84c59962f04daeb1234579bc3b9afa-0": {},
        "39d113d5868d473caf71e928d1d313a6-0": {},
        "c2b22722e149416bacf7420a59cb472b-0": {},
        "c2b22722e149416bacf7420a59cb472b-1": {},
        "2444ea72c6c84f91a4a6505206c7a9d3-0": {},
        "2444ea72c6c84f91a4a6505206c7a9d3-1": {},
        "fb0a0301abf6431485b70e497aa3f5bc-0": {},
        "341fbd5544f541bbb4d5edf5007c236d-0": {},
        "341fbd5544f541bbb4d5edf5007c236d-1": {},
        "fac6dd9a59c047c5a8c6cf4d7de4dedb-0": {}
    }
}

# 保存配置
with open('scene_config.json', 'w') as f:
    json.dump(scene_config, f, indent=2)

print('\n✅ 场景配置已创建')
print('\n📄 配置内容:')
print(json.dumps(scene_config, indent=2))

print('\n' + '='*60)
print('💡 提示: 你可以查看 benchmark_tasks/ 目录')
print('   获取更多场景配置示例')
print('='*60)

# 显示可用的示例
print('\n📚 可用的benchmark示例:')
!find benchmark_tasks -name "*.json" | head -5

📝 创建场景配置

✅ 场景配置已创建

📄 配置内容:
{
  "task_description": "a beach-inspired bedroom with light paint, a rattan chair and a queen bed, 4m x 5m",
  "layout_criteria": "airy, open, and serene with a focus on natural light and maximizing the feeling of space.\n",
  "boundary": {
    "floor_vertices": [
      [
        0,
        0,
        0
      ],
      [
        4,
        0,
        0
      ],
      [
        4,
        5,
        0
      ],
      [
        0,
        5,
        0
      ]
    ],
    "wall_height": 1.8687612077960416
  },
  "assets": {
    "ed84c59962f04daeb1234579bc3b9afa-0": {},
    "39d113d5868d473caf71e928d1d313a6-0": {},
    "c2b22722e149416bacf7420a59cb472b-0": {},
    "c2b22722e149416bacf7420a59cb472b-1": {},
    "2444ea72c6c84f91a4a6505206c7a9d3-0": {},
    "2444ea72c6c84f91a4a6505206c7a9d3-1": {},
    "fb0a0301abf6431485b70e497aa3f5bc-0": {},
    "341fbd5544f541bbb4d5edf5007c236d-0": {},
    "341fbd5544f541bbb4d5edf5007c236d-1": {},
    "fac6dd9a59c047c5a8c6cf4d7de

## 步骤 1️⃣1️⃣: 运行LayoutVLM

🚀 **核心步骤**: 使用Blender Python运行LayoutVLM生成布局

⏱️ 预计时间: 5-15分钟（取决于场景复杂度）

In [ ]:
import os

BLENDER = os.environ['BLENDER_PATH']
os.chdir('/content/LayoutVLM')

print('='*60)
print('🚀 运行LayoutVLM')
print('='*60)
print('\n⏱️  这可能需要5-15分钟，请耐心等待...')
print('💡 你可以在下方看到实时进度\n')

# 创建运行脚本
run_script = f'''import os
import sys

# ⚠️ 关键修复：在导入任何模块前设置matplotlib后端
os.environ["MPLBACKEND"] = "Agg"  # 使用非交互式后端

# 设置环境变量
os.environ["OPENAI_API_KEY"] = "{API_KEY}"
os.environ["OPENAI_BASE_URL"] = "{BASE_URL}"

# 添加路径
sys.path.insert(0, "/content/LayoutVLM")

# 设置命令行参数
sys.argv = [
    "main.py",
    "--scene_json_file", "scene_config.json",
    "--asset_dir", "/content/LayoutVLM/data/test_asset_dir",
    "--openai_api_key", "{API_KEY}",
    "--openai_base_url", "{BASE_URL}",  # ← 关键！这是之前缺少的
    "--save_dir", "{PROJECT_DIR}/results"
]

print("="*60)
print("🎨 LayoutVLM 开始生成布局")
print("="*60)
print()

# 执行main.py
try:
    with open("/content/LayoutVLM/main.py", "r") as f:
        exec(f.read())
    print()
    print("="*60)
    print("✅ LayoutVLM执行完成")
    print("="*60)
except Exception as e:
    print()
    print("="*60)
    print(f"❌ 执行出错: {{e}}")
    print("="*60)
    import traceback
    traceback.print_exc()
'''

# 保存运行脚本
with open('/tmp/run_layoutvlm.py', 'w') as f:
    f.write(run_script)

print('='*60)
print('开始执行...')
print('='*60)
print()

# 使用xvfb和Blender运行
!xvfb-run -a $BLENDER --background --python /tmp/run_layoutvlm.py

print()
print('='*60)
print('✅ 运行完成！')
print('='*60)
print(f'📁 结果保存在: {PROJECT_DIR}/results')
print('='*60)

🚀 运行LayoutVLM

⏱️  这可能需要5-15分钟，请耐心等待...
💡 你可以在下方看到实时进度

开始执行...

Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)
🎨 LayoutVLM 开始生成布局

/content/LayoutVLM/src/layoutvlm/constraints.py:15: UserWarning: WARNING: Could not import oriented_iou_loss from third_party.Rotated_IoU. This is likely due to CUDA not being available or the CUDA extension not being compiled. The bbox_overlap_loss function will use a simplified fallback that may be less accurate. Original error: No module named 'box_intersection_2d'
  warnings.warn(
Placing unplaced assets -- group 0
Info: Deleted 1 data-block(s)
White background set up.
No texture found for the object.
HDRI file not found: /content/LayoutVLM/./data/HDRIs/studio_small_08_4k.exr
Fra:1 Mem:11.27M (Peak 11.27M) | Time:00:00.00 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | X_Axis_tip
Fra:1 Mem:11.27M (Peak 11.27M) | Time:00:00.00 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | Z_Axis_tip
Fra:1 Mem:11

In [25]:
# ========== 快速诊断 ==========

import os
import glob

data_dir = '/content/LayoutVLM/data'

print('='*60)
print('📊 数据集快速诊断')
print('='*60)

# 1. 目录大小
!du -sh {data_dir}

# 2. 文件数量
print('\n文件统计:')
!find {data_dir} -type f | wc -l

# 3. 文件类型分布
print('\n文件类型:')
!find {data_dir} -type f -name "*.*" | sed 's/.*\.//' | sort | uniq -c | sort -rn | head -10

# 4. 目录结构
print('\n目录结构 (前2层):')
!tree {data_dir} -L 2 -d 2>/dev/null || find {data_dir} -maxdepth 2 -type d | head -20

# 5. 查找JSON文件
print('\n数据集中的JSON文件:')
!find {data_dir} -name "*.json" -type f | head -10

📊 数据集快速诊断
3.7G	/content/LayoutVLM/data

文件统计:
6041

文件类型:
   2696 jpg
   1160 png
    674 json
    674 gz
    674 glb
     82 obj
     43 mtl
     38 urdf

目录结构 (前2层):
/content/LayoutVLM/data
/content/LayoutVLM/data/test_asset_dir
/content/LayoutVLM/data/test_asset_dir/e72a40a1adde489b9e0149f94eb7d967
/content/LayoutVLM/data/test_asset_dir/1e835c246ba34627a012f9d6eb2b149d
/content/LayoutVLM/data/test_asset_dir/f6a8d9cf57bb4342840e790d02d1043d
/content/LayoutVLM/data/test_asset_dir/ac4b100dda264a189c837b784a8e242e
/content/LayoutVLM/data/test_asset_dir/2ba80c2563dd4d03a5f719caa0bd1f1c
/content/LayoutVLM/data/test_asset_dir/e57094e36af6477b9b4b1e3307f7a8a8
/content/LayoutVLM/data/test_asset_dir/fc0075723a0a4d7e93aee8642503e171
/content/LayoutVLM/data/test_asset_dir/999f7b6bb4a44a6e933aeb59e68c38ef
/content/LayoutVLM/data/test_asset_dir/7a09f0a44966446183c126358f12b08e
/content/LayoutVLM/data/test_asset_dir/5fdd2647427b4543bd299acff008bc3f
/content/LayoutVLM/data/test_asset_dir/b081286782

In [29]:
import os
img_path = "/content/drive/MyDrive/LayoutVLM_Project/results/group_0/side_rendering_45_3.png"
print(f"Image size: {os.path.getsize(img_path) / 1024 / 1024:.2f} MB")

Image size: 0.40 MB


## 步骤 1️⃣2️⃣: 查看生成结果

📊 查看生成的布局数据和渲染图片

In [26]:
import os
import json
from IPython.display import Image, display
import glob

result_dir = f'{PROJECT_DIR}/results'

print('='*60)
print('📊 查看生成结果')
print('='*60)

# 列出所有生成的文件
print('\n📁 生成的文件:')
!ls -lh {result_dir}

# 读取布局JSON
layout_file = f'{result_dir}/layout.json'

if os.path.exists(layout_file):
    print('\n' + '='*60)
    print('📋 布局数据 (layout.json):')
    print('='*60)

    with open(layout_file, 'r') as f:
        layout = json.load(f)

    # 显示简要信息
    print(f'\n✅ 成功生成 {len(layout)} 个物体的布局')

    # 显示前几个物体的信息
    print('\n前3个物体的信息:')
    for i, (obj_id, obj_info) in enumerate(list(layout.items())[:3], 1):
        print(f'\n{i}. {obj_id}:')
        print(f'   位置: {obj_info.get("position", "N/A")}')
        print(f'   旋转: {obj_info.get("rotation", "N/A")}')
        if len(layout) > 3 and i == 3:
            print(f'\n... 还有 {len(layout) - 3} 个物体')

    # 完整JSON预览
    print('\n完整JSON数据（前500字符）:')
    json_str = json.dumps(layout, indent=2)
    print(json_str[:500])
    if len(json_str) > 500:
        print(f'\n... (还有 {len(json_str) - 500} 字符)')

else:
    print('\n❌ 未找到 layout.json 文件')
    print('   请检查运行日志中的错误信息')

# 查找并显示渲染图片
print('\n' + '='*60)
print('🖼️  渲染图片:')
print('='*60)

image_files = glob.glob(f'{result_dir}/*.png') + glob.glob(f'{result_dir}/*.jpg')

if image_files:
    print(f'\n找到 {len(image_files)} 张图片:\n')
    for img_path in image_files[:5]:  # 最多显示5张
        print(f'📷 {os.path.basename(img_path)}')
        try:
            display(Image(filename=img_path, width=600))
            print()
        except:
            print(f'   ⚠️  无法显示图片')

    if len(image_files) > 5:
        print(f'\n... 还有 {len(image_files) - 5} 张图片')
else:
    print('\n⚠️  未找到渲染图片')
    print('   可能渲染功能未启用或发生错误')

print('\n' + '='*60)
print('✅ 结果查看完成')
print('='*60)
print(f'\n💾 所有结果已保存在: {result_dir}')
print('   你可以在Google Drive中查看完整结果')

📊 查看生成结果

📁 生成的文件:
total 17K
-rw------- 1 root root  12K Nov  3 06:50 complete_sandbox_program.py
drwx------ 2 root root 4.0K Nov  3 06:51 group_0
-rw------- 1 root root    2 Nov  2 06:31 layout.json

📋 布局数据 (layout.json):

✅ 成功生成 0 个物体的布局

前3个物体的信息:

完整JSON数据（前500字符）:
{}

🖼️  渲染图片:

⚠️  未找到渲染图片
   可能渲染功能未启用或发生错误

✅ 结果查看完成

💾 所有结果已保存在: /content/drive/MyDrive/LayoutVLM_Project/results
   你可以在Google Drive中查看完整结果


---

## 🔧 故障排查

### 常见问题

#### 1. API连接失败
- 检查API Key是否正确
- 确认Base URL格式正确（包含 `/v1/`）
- 检查账户余额

#### 2. 内存不足
- 尝试使用高RAM运行时
- 减小场景复杂度

#### 3. GPU相关错误
- 确认已启用GPU运行时
- 检查CUDA版本兼容性

#### 4. 依赖安装失败
- 重新运行步骤6
- 检查网络连接

### 重新运行

如果需要重新运行：
1. **完全重启**: 运行时 → 重启运行时
2. **从头执行**: 按顺序重新运行所有单元格
3. **数据集会从Drive快速恢复**（无需重新下载）

---

## 📚 参考资源

- **论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)
- **GitHub**: https://github.com/sunfanyunn/LayoutVLM
- **项目主页**: https://ai.stanford.edu/~sunfanyun/layoutvlm/
- **CVPR 2025**: Proceedings (June 2025)

---

## 💡 下一步

1. **尝试不同场景**: 修改`scene_config.json`
2. **使用benchmark**: 复制`benchmark_tasks`中的示例
3. **调整参数**: 修改布局标准和边界
4. **自定义资产**: 添加自己的3D模型

---